In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/processed/full_dataset.csv')
print(df.shape)
df.head()

(3416, 16)


,Abbreviation,TeamName,GridPosition,Position,Points,Status,Q1,Q2,Q3,BestQualiTime,QualiGapToPole,Year,Race,Round,DNF,Podium
0,VER,Red Bull Racing,1.0,1.0,25.0,Finished,0 days 00:01:31.295000,0 days 00:01:30.503000,0 days 00:01:29.708000,89.708,0.000,2023,Bahrain Grand Prix,1,0,1
1,PER,Red Bull Racing,2.0,2.0,18.0,Finished,0 days 00:01:31.479000,0 days 00:01:30.746000,0 days 00:01:29.846000,89.846,0.138,2023,Bahrain Grand Prix,1,0,1
2,ALO,Aston Martin,5.0,3.0,15.0,Finished,0 days 00:01:31.158000,0 days 00:01:30.645000,0 days 00:01:30.336000,90.336,0.628,2023,Bahrain Grand Prix,1,0,1
3,SAI,Ferrari,4.0,4.0,12.0,Finished,0 days 00:01:30.993000,0 days 00:01:30.515000,0 days 00:01:30.154000,90.154,0.446,2023,Bahrain Grand Prix,1,0,0
4,HAM,Mercedes,7.0,5.0,10.0,Finished,0 days 00:01:31.543000,0 days 00:01:30.513000,0 days 00:01:30.384000,90.384,0.676,2023,Bahrain Grand Prix,1,0,0


In [35]:
# Rolling average finish — last 3 and 5 races
df['AvgFinish_Last3'] = (
    df.groupby('Abbreviation')['Position']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

df['AvgFinish_Last5'] = (
    df.groupby('Abbreviation')['Position']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

# DNF rate last 5 races
df['DNFRate_Last5'] = (
    df.groupby('Abbreviation')['DNF']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

print(df[['Abbreviation', 'Race', 'Year', 'Position', 
          'AvgFinish_Last3', 'AvgFinish_Last5', 'DNFRate_Last5']].head(20))

   Abbreviation                         Race  Year  Position  AvgFinish_Last3  \
0           AIT            Sakhir Grand Prix  2020      16.0              NaN   
1           ALB          Austrian Grand Prix  2020      13.0              NaN   
2           ALB           Styrian Grand Prix  2020       4.0        13.000000   
3           ALB         Hungarian Grand Prix  2020       5.0         8.500000   
4           ALB           British Grand Prix  2020       8.0         7.333333   
5           ALB  70th Anniversary Grand Prix  2020       5.0         5.666667   
6           ALB           Spanish Grand Prix  2020       8.0         6.000000   
7           ALB           Belgian Grand Prix  2020       6.0         7.000000   
8           ALB           Italian Grand Prix  2020      15.0         6.333333   
9           ALB            Tuscan Grand Prix  2020       3.0         9.666667   
10          ALB           Russian Grand Prix  2020      10.0         8.000000   
11          ALB             

In [2]:
import os
print(os.getcwd())
os.chdir(r'C:\Users\91917\OneDrive\Desktop\f1-predictor')
print(os.getcwd())

c:\Users\91917\OneDrive\Desktop\f1-predictor\notebooks
C:\Users\91917\OneDrive\Desktop\f1-predictor


In [4]:
# Cell 2 — Basic cleaning
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')
df['GridPosition'] = pd.to_numeric(df['GridPosition'], errors='coerce')

# Drop rows where Position is null (e.g. DNS)
df = df.dropna(subset=['Position', 'GridPosition'])
print(f"Clean shape: {df.shape}")

Clean shape: (3393, 16)


In [5]:
df['GridToFinishDelta'] = df['GridPosition'] - df['Position']
# Positive = gained places, negative = lost places

df['NormalizedPosition'] = df.groupby(['Year', 'Race'])['Position'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)
# 0 = winner, 1 = last — normalized within each race

In [6]:
team_avg = df.groupby(['Year', 'Race', 'TeamName'])['Position'].median().reset_index()
team_avg.columns = ['Year', 'Race', 'TeamName', 'TeamMedianPosition']

df = df.merge(team_avg, on=['Year', 'Race', 'TeamName'], how='left')

df['DriverVsTeamDelta'] = df['TeamMedianPosition'] - df['Position']

In [7]:
import os

for root, dirs, files in os.walk('data/'):
    for file in files:
        path = os.path.join(root, file)
        print(path)

data/processed\driver_skill.csv
data/processed\driver_skill_features.csv
data/processed\final_model.pkl
data/processed\full_dataset.csv
data/processed\season_2020.csv
data/processed\season_2021.csv
data/processed\season_2022.csv
data/processed\season_2023.csv
data/processed\season_2024.csv
data/processed\season_2025.csv
data/raw\f1_raw.csv


In [8]:
season_2023 = pd.read_csv('data/processed/season_2023.csv')
print(f"2023 rows: {season_2023.shape[0]}")

# Combine
full_data = pd.concat([df, season_2023], ignore_index=True)

# Remove any duplicates just in case
full_data = full_data.drop_duplicates(subset=['Abbreviation', 'Year', 'Race'])

# Save
full_data.to_csv('data/processed/full_dataset.csv', index=False)
print(f"Final dataset: {full_data.shape[0]} rows")

# Reload
df = full_data.copy()
print(df['Year'].value_counts().sort_index())

2023 rows: 440
Final dataset: 2596 rows
Year
2020    340
2021    419
2022    439
2023    440
2024    479
2025    479
Name: count, dtype: int64


In [10]:
import os
for f in os.listdir('data/processed/'):
    print(f)

driver_skill.csv
driver_skill_features.csv
final_model.pkl
full_dataset.csv
season_2020.csv
season_2021.csv
season_2022.csv
season_2023.csv
season_2024.csv
season_2025.csv


In [12]:
import pandas as pd

s2020 = pd.read_csv('data/processed/season_2020.csv')
s2021 = pd.read_csv('data/processed/season_2021.csv')
s2022 = pd.read_csv('data/processed/season_2022.csv')
s2023 = pd.read_csv('data/processed/season_2023.csv')

print(f"2020: {s2020.shape[0]} rows")
print(f"2021: {s2021.shape[0]} rows")
print(f"2022: {s2022.shape[0]} rows")
print(f"2023: {s2023.shape[0]} rows")
print(f"2024: {s2024.shape[0]} rows")
print(f"2025: {s2025.shape[0]} rows")

# Combine all
df = pd.concat([s2020, s2021, s2022, s2023], ignore_index=True)
df = df.drop_duplicates(subset=['Abbreviation', 'Year', 'Race'])

# Save
df.to_csv('data/processed/full_dataset.csv', index=False)
print(f"\nFinal dataset: {df.shape[0]} rows")
print(df['Year'].value_counts().sort_index())

2020: 340 rows
2021: 440 rows
2022: 440 rows
2023: 440 rows


NameError: name 's2024' is not defined

In [11]:
print(df['Year'].value_counts().sort_index())
print(f"\nRaces per season:")
print(df.groupby('Year')['Race'].nunique())

Year
2020    340
2021    440
2022    440
2023    440
Name: count, dtype: int64

Races per season:
Year
2020    17
2021    22
2022    22
2023    22
Name: Race, dtype: int64


In [13]:
df = df.sort_values(['Abbreviation', 'Year', 'Round'])

df['SeasonPointsSoFar'] = (
    df.groupby(['Abbreviation', 'Year'])['Points']
    .transform(lambda x: x.shift(1).cumsum().fillna(0))
)

In [14]:
df['TeamSeasonPoints'] = (
    df.groupby(['TeamName', 'Year'])['Points']
    .transform(lambda x: x.shift(1).cumsum().fillna(0))
)

In [15]:
skill_rows = []

for (year, race, team), group in df.groupby(['Year', 'Race', 'TeamName']):
    if len(group) < 2:
        continue
    
    drivers = group[['Abbreviation', 'Position', 'QualiGapToPole']].values
    
    for i in range(len(drivers)):
        for j in range(len(drivers)):
            if i == j:
                continue
            
            driver = drivers[i]
            teammate = drivers[j]
            
            driver_dnf = group[group['Abbreviation'] == driver[0]]['DNF'].values[0]
            teammate_dnf = group[group['Abbreviation'] == teammate[0]]['DNF'].values[0]
            if driver_dnf == 1 or teammate_dnf == 1:
                continue
            
            pos_delta = teammate[1] - driver[1]
            quali_delta = teammate[2] - driver[2]
            
            skill_rows.append({
                'Year': year,
                'Race': race,
                'Abbreviation': driver[0],
                'Teammate': teammate[0],
                'TeamName': team,
                'PositionDelta': pos_delta,
                'QualiDelta': quali_delta
            })

skill_df = pd.DataFrame(skill_rows)
print(f"Skill comparisons: {skill_df.shape[0]}")
skill_df.head(10)

Skill comparisons: 1460


,Year,Race,Abbreviation,Teammate,TeamName,PositionDelta,QualiDelta
0,2020,70th Anniversary Grand Prix,GIO,RAI,Alfa Romeo Racing,-2.0,0.060
1,2020,70th Anniversary Grand Prix,RAI,GIO,Alfa Romeo Racing,2.0,-0.060
2,2020,70th Anniversary Grand Prix,GAS,KVY,AlphaTauri,-1.0,1.348
3,2020,70th Anniversary Grand Prix,KVY,GAS,AlphaTauri,1.0,-1.348
4,2020,70th Anniversary Grand Prix,LEC,VET,Ferrari,8.0,0.464
5,2020,70th Anniversary Grand Prix,VET,LEC,Ferrari,-8.0,-0.464
6,2020,70th Anniversary Grand Prix,NOR,SAI,McLaren,4.0,0.305
7,2020,70th Anniversary Grand Prix,SAI,NOR,McLaren,-4.0,-0.305
8,2020,70th Anniversary Grand Prix,BOT,HAM,Mercedes,-1.0,0.063
9,2020,70th Anniversary Grand Prix,HAM,BOT,Mercedes,1.0,-0.063


In [22]:
driver_skill = skill_df.groupby(['Year', 'Abbreviation']).agg(
    AvgPositionDelta=('PositionDelta', 'mean'),
    AvgQualiDelta=('QualiDelta', 'mean'),
    Races=('PositionDelta', 'count')
).reset_index()

# Normalize to 0-100
min_val = driver_skill['AvgPositionDelta'].min()
max_val = driver_skill['AvgPositionDelta'].max()

driver_skill['SkillScore'] = (
    (driver_skill['AvgPositionDelta'] - min_val) / 
    (max_val - min_val) * 100
).round(1)

# Show 2023 rankings
print("=== 2023 Driver Skill Rankings ===")
print(driver_skill[driver_skill['Year'] == 2023]
      .sort_values('SkillScore', ascending=False)
      [['Abbreviation', 'AvgPositionDelta', 'SkillScore', 'Races']]
      .to_string(index=False))

=== 2023 Driver Skill Rankings ===
Abbreviation  AvgPositionDelta  SkillScore  Races
         VER          3.952381        81.9     21
         ALO          3.470588        77.7     17
         ALB          3.384615        76.9     13
         LAW          2.500000        69.1      4
         NOR          2.388889        68.1     18
         SAI          1.388889        59.3     18
         TSU          1.000000        55.9     18
         OCO          0.866667        54.7     15
         BOT          0.470588        51.2     17
         HAM          0.294118        49.7     17
         HUL          0.066667        47.6     15
         MAG         -0.066667        46.5     15
         RUS         -0.294118        44.5     17
         ZHO         -0.470588        42.9     17
         GAS         -0.866667        39.4     15
         LEC         -1.388889        34.8     18
         DEV         -1.875000        30.5      8
         RIC         -2.166667        27.9      6
         PIA   

In [23]:
driver_skill.to_csv('data/processed/driver_skill.csv', index=False)
print("Saved ✓")

Saved ✓


In [24]:
print("=== 2021 Driver Skill Rankings ===")
print(driver_skill[driver_skill['Year'] == 2021]
      .sort_values('SkillScore', ascending=False)
      [['Abbreviation', 'AvgPositionDelta', 'SkillScore', 'Races']]
      .to_string(index=False))

=== 2021 Driver Skill Rankings ===
Abbreviation  AvgPositionDelta  SkillScore  Races
         GAS          4.380952        85.7     21
         VER          4.380952        85.7     21
         HAM          4.000000        82.4     22
         NOR          1.590909        61.1     22
         RUS          1.315789        58.7     19
         SAI          1.227273        57.9     22
         MSC          0.684211        53.1     19
         ALO          0.666667        52.9     21
         STR          0.400000        50.6     20
         GIO          0.227273        49.1     22
         RAI         -0.150000        45.7     20
         VET         -0.400000        43.5     20
         OCO         -0.666667        41.2     21
         MAZ         -0.684211        41.0     19
         KUB         -1.000000        38.2      2
         LEC         -1.227273        36.2     22
         LAT         -1.315789        35.4     19
         RIC         -1.590909        33.0     22
         BOT   

In [25]:
print(driver_skill['Year'].value_counts().sort_index())

Year
2020    23
2021    21
2022    22
2023    22
Name: count, dtype: int64


In [ ]:
df = df.merge(
    driver_skill[['Year', 'Abbreviation', 'SkillScore', 'AvgQualiDelta']],
    on=['Year', 'Abbreviation'],
    how='left'
)

print(df.shape)
print(df[['Abbreviation', 'Year', 'Race', 'Position', 'SkillScore']].head(10))

In [38]:
df.to_csv('data/processed/driver_skill_features.csv', index=False)
print(f"Saved: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Saved: (1637, 29)
Columns: ['Abbreviation', 'TeamName', 'GridPosition', 'Position', 'Points', 'Status', 'Q1', 'Q2', 'Q3', 'BestQualiTime', 'QualiGapToPole', 'Year', 'Race', 'Round', 'DNF', 'Podium', 'SeasonPointsSoFar', 'TeamSeasonPoints', 'SkillScore_x', 'AvgQualiDelta_x', 'GridToFinishDelta', 'NormalizedPosition', 'TeamMedianPosition', 'DriverVsTeamDelta', 'AvgFinish_Last3', 'AvgFinish_Last5', 'DNFRate_Last5', 'SkillScore_y', 'AvgQualiDelta_y']


In [39]:
print(df.columns.tolist())

['Abbreviation', 'TeamName', 'GridPosition', 'Position', 'Points', 'Status', 'Q1', 'Q2', 'Q3', 'BestQualiTime', 'QualiGapToPole', 'Year', 'Race', 'Round', 'DNF', 'Podium', 'SeasonPointsSoFar', 'TeamSeasonPoints', 'SkillScore_x', 'AvgQualiDelta_x', 'GridToFinishDelta', 'NormalizedPosition', 'TeamMedianPosition', 'DriverVsTeamDelta', 'AvgFinish_Last3', 'AvgFinish_Last5', 'DNFRate_Last5', 'SkillScore_y', 'AvgQualiDelta_y']


In [40]:
# Reload clean base
df = pd.read_csv('data/processed/driver_skill_features.csv')

# Drop duplicate skill columns
df = df.drop(columns=['SkillScore_y', 'AvgQualiDelta_y'])
df = df.rename(columns={'SkillScore_x': 'SkillScore', 'AvgQualiDelta_x': 'AvgQualiDelta'})

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape: (1637, 27)
Columns: ['Abbreviation', 'TeamName', 'GridPosition', 'Position', 'Points', 'Status', 'Q1', 'Q2', 'Q3', 'BestQualiTime', 'QualiGapToPole', 'Year', 'Race', 'Round', 'DNF', 'Podium', 'SeasonPointsSoFar', 'TeamSeasonPoints', 'SkillScore', 'AvgQualiDelta', 'GridToFinishDelta', 'NormalizedPosition', 'TeamMedianPosition', 'DriverVsTeamDelta', 'AvgFinish_Last3', 'AvgFinish_Last5', 'DNFRate_Last5']


In [41]:
df.to_csv('data/processed/driver_skill_features.csv', index=False)
print("Saved clean ✓")

Saved clean ✓


In [44]:
street_circuits = [
    'Monaco Grand Prix', 'Azerbaijan Grand Prix',
    'Singapore Grand Prix', 'Saudi Arabian Grand Prix',
    'Miami Grand Prix', 'Las Vegas Grand Prix',
    'Abu Dhabi Grand Prix'
]

df['IsStreetCircuit'] = df['Race'].isin(street_circuits).astype(int)

# Constructor championship gap — how dominant is this team right now
df['ConstructorGap'] = df.groupby(['Year', 'Race'])['TeamSeasonPoints'].transform(
    lambda x: x.max() - x
)

# Driver championship position proxy
df['DriverPointsRank'] = df.groupby(['Year', 'Race'])['SeasonPointsSoFar'].rank(ascending=False)

# Save
df.to_csv('data/processed/driver_skill_features.csv', index=False)
print("Features added ✓")

Features added ✓
